# 06 — Diseño multivariado de sensor virtual VIS_COR

## Proyecto
**Kaki_DeepHS_StorageDays**

## Pregunta

El notebook 05 mostró que seleccionar bandas individuales por monotonía temporal no conserva la precisión del espectro completo, aunque los paneles pequeños mejoraron la robustez `session-excluded`.

Aquí se cambia la pregunta:

> ¿Puede diseñarse un sensor multiespectral pequeño seleccionando **canales virtuales multivariados** directamente por desempeño `fruit-disjoint`?

## Cambio metodológico principal

En vez de seleccionar columnas individuales de la cámara HSI, se simulan filtros espectrales gaussianos:

\[
z_j = \sum_\lambda w_j(\lambda)R(\lambda)
\]

con anchos espectrales:

- FWHM = 10 nm
- FWHM = 20 nm

y centros candidatos aproximadamente cada 10 nm dentro del rango VIS_COR.

Esto se aproxima mejor a un sensor multiespectral real que escoger bandas HSI infinitamente estrechas.

## Selección multivariada

Dentro de cada outer fold:

1. 8 frutos quedan completamente aislados como test.
2. Con los 32 frutos de desarrollo se ejecuta **forward selection**.
3. En cada paso se prueba cada nuevo filtro candidato.
4. El candidato elegido es el que minimiza `MAE fruit-balanced` mediante CV interna fruit-disjoint.
5. Se construye una ruta hasta 10 canales.
6. Se evalúan los prefijos:

\[
k \in \{3,5,8,10\}
\]

7. Para cada panel se ajustan `RAW/SNV + Ridge(alpha)` solo dentro del desarrollo.
8. El test externo se usa una sola vez.

## Stress test session-excluded

Se conserva el panel seleccionado con los 32 frutos de desarrollo y se vuelve a ajustar el modelo excluyendo el día/sesión objetivo.

El null correcto es `SessionExcluded_DummyMean`, que sufre la misma exclusión.

### Limitación

El panel virtual usado en el stress test fue diseñado observando todos los días de los frutos de desarrollo. Por tanto, `session-excluded` evalúa robustez del **modelo con un panel fijo**, pero no elimina la exposición de la etapa de diseño del sensor a las sesiones del desarrollo.

## Comparaciones

- Full VIS_COR 249 bandas — notebook 02.
- Full VIS_COR session-excluded — notebook 03.
- Selector univariado del notebook 05, si está disponible.
- Sensor virtual multivariado de 3/5/8/10 canales.
- Null estándar y null session-excluded.

No se entrenan CNN en este notebook.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import math
import platform
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "06_kaki_multivariate_virtual_sensor_design"
RUN_SIGNATURE = "viscor_virtual_filters_forward_selection_v1"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

BASE02 = ROOT / "results/02_kaki_fruit_disjoint_splits_baselines"
BASE03 = ROOT / "results/03_kaki_robustness_session_bias"
BASE05 = ROOT / "results/05_kaki_wavelength_selection_sensor_simplification"

RESULTS = ROOT / "results/06_kaki_multivariate_virtual_sensor_design"
FIGURES = ROOT / "figures/06_kaki_multivariate_virtual_sensor_design"
TABLES = ROOT / "tables/06_kaki_multivariate_virtual_sensor_design"
EXPORT_ROOT = ROOT / "exports/06_kaki_multivariate_virtual_sensor_design"
CHECKPOINT_DIR = RESULTS / f"checkpoints_{RUN_SIGNATURE}"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
    CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

CAMERA = "VIS_COR"

FWHM_VALUES = [
    10.0,
    20.0,
]

CENTER_STEP_NM = 10.0

K_VALUES = [
    3,
    5,
    8,
    10,
]

MAX_K = max(
    K_VALUES
)

# En forward selection se usa RAW + escalado y una grilla
# pequeña de alpha para mantener el cómputo razonable.
SELECTION_ALPHAS = [
    0.1,
    1.0,
    10.0,
    100.0,
]

# Una vez fijado el panel, se hace tuning final más completo.
FINAL_PREPROCESSING = [
    "raw",
    "snv",
]

FINAL_ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

OUTER_FOLDS = 5
INNER_FOLDS = 4

RANDOM_SEED = 20260818
BOOTSTRAP_REPS = 5000

RESUME = True

# Pantalla descriptiva, NO prueba formal de equivalencia.
ENGINEERING_DELTA_DAYS = 0.10

print("ROOT:", ROOT)
print("FWHM:", FWHM_VALUES)
print("k:", K_VALUES)
print("RESUME:", RESUME)


## 2. Cargar VIS_COR, splits congelados y referencias


In [ ]:
SPECTRA_PATH = (
    INTERIM /
    "kaki_fruit_day_spectra_VIS_COR.csv"
)

CUBE_PATH = (
    INTERIM /
    "kaki_cube_spectra_VIS_COR.csv"
)

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

BASELINE_OOF_PATH = (
    BASE02 /
    "kaki_oof_predictions.csv"
)

SESSION_OOF_PATH = (
    BASE03 /
    "kaki_session_excluded_oof_predictions.csv"
)

OPTIONAL_05_STANDARD = (
    BASE05 /
    "kaki_viscor_sparse_standard_oof.csv"
)

OPTIONAL_05_SESSION = (
    BASE05 /
    "kaki_viscor_sparse_session_excluded_oof.csv"
)

for p in [
    SPECTRA_PATH,
    CUBE_PATH,
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
    BASELINE_OOF_PATH,
    SESSION_OOF_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo requerido: {p}"
        )

cube = pd.read_csv(
    CUBE_PATH
)

assert len(cube) == 398
assert float(
    cube[
        "mask_coverage"
    ].median()
) < 0.87
assert (
    cube[
        "mask_qc_flag"
    ]
    ==
    "ok"
).all()
assert cube[
    "extraction_error"
].isna().all()

data = pd.read_csv(
    SPECTRA_PATH
)

data["fruit_id"] = pd.to_numeric(
    data[
        "fruit_id"
    ],
    errors="raise"
).astype(int)

data["storage_days_model"] = pd.to_numeric(
    data[
        "storage_days_model"
    ],
    errors="raise"
).astype(int)

wl_cols = [
    c
    for c in data.columns
    if c.startswith(
        "wl_"
    )
]

wavelengths = np.array(
    [
        float(
            c.replace(
                "wl_",
                ""
            )
        )
        for c in wl_cols
    ],
    dtype=float
)

assert len(data) == 199
assert data["fruit_id"].nunique() == 40
assert len(wl_cols) == 249
assert not data[
    wl_cols
].isna().any().any()

data = data.sort_values(
    [
        "fruit_id",
        "storage_days_model"
    ]
).reset_index(
    drop=True
)

outer = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner = pd.read_csv(
    INNER_SPLIT_PATH
)

baseline_oof = pd.read_csv(
    BASELINE_OOF_PATH
)

full_session_oof_raw = pd.read_csv(
    SESSION_OOF_PATH
)

for df in [
    outer,
    inner,
]:
    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

outer[
    "outer_fold"
] = pd.to_numeric(
    outer[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "outer_fold"
] = pd.to_numeric(
    inner[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "inner_fold"
] = pd.to_numeric(
    inner[
        "inner_fold"
    ],
    errors="raise"
).astype(int)

print(
    "VIS_COR:",
    len(data),
    "fruit-days |",
    data[
        "fruit_id"
    ].nunique(),
    "frutos |",
    len(
        wavelengths
    ),
    "bandas"
)

print(
    f"Rango espectral: "
    f"{wavelengths.min():.3f}–"
    f"{wavelengths.max():.3f} nm"
)

print("✅ Inputs cargados.")


## 3. Auditoría anti-leakage


In [ ]:
audit_rows = []

all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    train_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner.loc[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    audit_rows.append({
        "outer_fold":
            int(
                outer_fold
            ),

        "n_train_fruits":
            len(
                train_fruits
            ),

        "n_test_fruits":
            len(
                test_fruits
            ),

        "train_test_overlap":
            len(
                train_fruits
                &
                test_fruits
            ),

        "inner_test_overlap":
            len(
                inner_fruits
                &
                test_fruits
            ),

        "inner_matches_train":
            inner_fruits
            ==
            train_fruits,
    })

split_audit = pd.DataFrame(
    audit_rows
)

display(
    split_audit
)

assert (
    split_audit[
        "train_test_overlap"
    ]
    ==
    0
).all()

assert (
    split_audit[
        "inner_test_overlap"
    ]
    ==
    0
).all()

assert split_audit[
    "inner_matches_train"
].all()

split_audit.to_csv(
    TABLES /
    "01_split_audit.csv",
    index=False
)

print("✅ Splits válidos.")


# 4. Construir bancos de filtros gaussianos virtuales

Para un filtro con centro \(c\) y ancho FWHM:

\[
\sigma =
\frac{\mathrm{FWHM}}
{2\sqrt{2\ln2}}
\]

\[
w(\lambda)
=
\exp
\left[
-\frac{1}{2}
\left(
\frac{\lambda-c}{\sigma}
\right)^2
\right]
\]

Los pesos se normalizan para sumar 1.

Los centros candidatos se ubican cada 10 nm. Se evitan filtros demasiado próximos al borde del rango espectral.


In [ ]:
def gaussian_filter_bank(
    wavelength_grid,
    fwhm_nm,
    center_step_nm=10.0
):

    wavelength_grid = np.asarray(
        wavelength_grid,
        dtype=float
    )

    sigma = (
        float(
            fwhm_nm
        )
        /
        (
            2.0
            *
            np.sqrt(
                2.0
                *
                np.log(
                    2.0
                )
            )
        )
    )

    # Mantener al menos ±1 FWHM dentro del rango.
    lower = (
        wavelength_grid.min()
        +
        float(
            fwhm_nm
        )
    )

    upper = (
        wavelength_grid.max()
        -
        float(
            fwhm_nm
        )
    )

    start = (
        np.ceil(
            lower
            /
            center_step_nm
        )
        *
        center_step_nm
    )

    stop = (
        np.floor(
            upper
            /
            center_step_nm
        )
        *
        center_step_nm
    )

    centers = np.arange(
        start,
        stop
        +
        0.5
        *
        center_step_nm,
        center_step_nm,
        dtype=float
    )

    weights = []

    meta_rows = []

    for center in centers:

        w = np.exp(
            -0.5
            *
            (
                (
                    wavelength_grid
                    -
                    center
                )
                /
                sigma
            ) ** 2
        )

        # Truncar aproximadamente en ±3 sigma.
        w[
            np.abs(
                wavelength_grid
                -
                center
            )
            >
            (
                3.0
                *
                sigma
            )
        ] = 0.0

        if (
            not np.isfinite(
                w
            ).all()
            or
            w.sum()
            <=
            0
        ):
            raise RuntimeError(
                "Filtro gaussiano inválido."
            )

        w = (
            w
            /
            w.sum()
        )

        weights.append(
            w
        )

        meta_rows.append({
            "candidate_id":
                (
                    f"FWHM{int(fwhm_nm)}_"
                    f"C{int(round(center))}"
                ),

            "center_nm":
                float(
                    center
                ),

            "fwhm_nm":
                float(
                    fwhm_nm
                ),

            "sigma_nm":
                float(
                    sigma
                ),

            "effective_min_nm":
                float(
                    wavelength_grid[
                        w > 0
                    ].min()
                ),

            "effective_max_nm":
                float(
                    wavelength_grid[
                        w > 0
                    ].max()
                ),
        })

    W = np.column_stack(
        weights
    )

    meta = pd.DataFrame(
        meta_rows
    )

    return (
        meta,
        W
    )


filter_banks = {}

for fwhm in FWHM_VALUES:

    meta, W = gaussian_filter_bank(
        wavelengths,
        fwhm_nm=fwhm,
        center_step_nm=CENTER_STEP_NM
    )

    filter_banks[
        fwhm
    ] = {
        "meta":
            meta,

        "weights":
            W,
    }

    print(
        f"FWHM {fwhm:.0f} nm:",
        len(
            meta
        ),
        "candidatos | centros",
        f"{meta['center_nm'].min():.0f}–"
        f"{meta['center_nm'].max():.0f} nm"
    )

print("✅ Bancos de filtros construidos.")


## 5. Visualizar respuestas espectrales de ejemplo


In [ ]:
for fwhm in FWHM_VALUES:

    meta = filter_banks[
        fwhm
    ][
        "meta"
    ]

    W = filter_banks[
        fwhm
    ][
        "weights"
    ]

    show_indices = np.linspace(
        0,
        len(
            meta
        )
        -
        1,
        8,
        dtype=int
    )

    fig, ax = plt.subplots(
        figsize=(10, 4.8)
    )

    for idx in show_indices:

        ax.plot(
            wavelengths,
            W[
                :,
                idx
            ],
            label=(
                f"{meta.iloc[idx]['center_nm']:.0f} nm"
            )
        )

    ax.set_xlabel(
        "Longitud de onda (nm)"
    )

    ax.set_ylabel(
        "Respuesta relativa normalizada"
    )

    ax.set_title(
        f"Filtros virtuales gaussianos — FWHM {fwhm:.0f} nm"
    )

    ax.legend(
        ncol=4,
        fontsize=8
    )

    ax.grid(
        alpha=0.15
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"01_virtual_filter_examples_fwhm{int(fwhm)}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 6. Precalcular las respuestas de todos los filtros

Así el forward selection trabaja sobre matrices pequeñas y no recalcula las convoluciones.


In [ ]:
X_full = data[
    wl_cols
].to_numpy(
    dtype=float
)

virtual_features = {}

for fwhm in FWHM_VALUES:

    W = filter_banks[
        fwhm
    ][
        "weights"
    ]

    Z = (
        X_full
        @
        W
    )

    meta = filter_banks[
        fwhm
    ][
        "meta"
    ]

    col_names = [
        str(
            x
        )
        for x in meta[
            "candidate_id"
        ]
    ]

    zdf = pd.DataFrame(
        Z,
        columns=col_names
    )

    zdf.insert(
        0,
        "storage_days_model",
        data[
            "storage_days_model"
        ].to_numpy(
            dtype=int
        )
    )

    zdf.insert(
        0,
        "fruit_id",
        data[
            "fruit_id"
        ].to_numpy(
            dtype=int
        )
    )

    virtual_features[
        fwhm
    ] = zdf

    assert len(
        zdf
    ) == 199

    assert not zdf[
        col_names
    ].isna().any().any()

    print(
        f"FWHM {fwhm:.0f}:",
        zdf.shape
    )

print("✅ Features virtuales listas.")


## 7. Funciones de métricas y Ridge


In [ ]:
def preprocess_panel(
    X,
    method
):

    X = np.asarray(
        X,
        dtype=float
    )

    if method == "raw":
        return X.copy()

    if method == "snv":

        if X.shape[
            1
        ] < 2:

            # SNV no tiene sentido con un único canal.
            return X.copy()

        mu = np.mean(
            X,
            axis=1,
            keepdims=True
        )

        sd = np.std(
            X,
            axis=1,
            keepdims=True
        )

        sd[
            (~np.isfinite(sd))
            |
            (sd == 0)
        ] = 1.0

        return (
            X
            -
            mu
        ) / sd

    raise ValueError(
        method
    )


def fit_predict_ridge(
    X_train,
    y_train,
    X_test,
    alpha
):

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(
        X_train
    )

    X_test_s = scaler.transform(
        X_test
    )

    model = Ridge(
        alpha=float(
            alpha
        )
    )

    model.fit(
        X_train_s,
        y_train
    )

    pred = model.predict(
        X_test_s
    )

    return np.asarray(
        pred,
        dtype=float
    )


def fruit_balanced_mae(
    prediction_df
):

    z = prediction_df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def day_balanced_mae(
    prediction_df
):

    z = prediction_df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "y_true"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def metric_bundle(
    prediction_df
):

    y_true = prediction_df[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    y_pred = prediction_df[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    residual = (
        y_pred
        -
        y_true
    )

    rho, _ = spearmanr(
        y_true,
        y_pred
    )

    return {
        "n_observations":
            int(
                len(
                    prediction_df
                )
            ),

        "n_fruits":
            int(
                prediction_df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y_true,
                    y_pred
                )
            ),

        "spearman_rho":
            float(
                rho
            ),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <=
                    1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                prediction_df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                prediction_df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


# 8. Score CV de un panel candidato

La selección forward usa únicamente:

- features RAW de los filtros;
- `StandardScaler` entrenado dentro del fold;
- Ridge;
- `alpha ∈ {0.1, 1, 10, 100}`.

Para cada panel candidato se escoge el mejor alpha mediante OOF interno, utilizando como métrica principal:

\[
MAE_{fruit-balanced}
\]

Esto hace que la elección del nuevo canal esté orientada directamente al desempeño multivariado en frutos no vistos dentro del desarrollo.


In [ ]:
def inner_cv_score_raw_panel(
    development_features,
    inner_assignment,
    selected_columns,
    alpha_grid=SELECTION_ALPHAS
):

    best = None

    rows = []

    for alpha in alpha_grid:

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_df = development_features[
                ~development_features[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            val_df = development_features[
                development_features[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            X_train = train_df[
                selected_columns
            ].to_numpy(
                dtype=float
            )

            X_val = val_df[
                selected_columns
            ].to_numpy(
                dtype=float
            )

            pred = fit_predict_ridge(
                X_train,
                train_df[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                X_val,
                alpha=alpha
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        val_df[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        val_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        pred_df = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            pred_df
        )

        db = day_balanced_mae(
            pred_df
        )

        mae = float(
            mean_absolute_error(
                pred_df[
                    "y_true"
                ],
                pred_df[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "alpha":
                float(
                    alpha
                ),

            "fruit_balanced_mae":
                fb,

            "day_balanced_mae":
                db,

            "mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),

                "fruit_balanced_mae":
                    fb,

                "day_balanced_mae":
                    db,

                "mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 9. Forward selection multivariada

Restricción de redundancia:

- FWHM 10 nm → centros seleccionados separados al menos 10 nm.
- FWHM 20 nm → centros seleccionados separados al menos 20 nm.

Esto evita escoger filtros esencialmente superpuestos.


In [ ]:
def forward_select_virtual_filters(
    development_features,
    inner_assignment,
    candidate_meta,
    max_k,
    min_center_spacing_nm
):

    selected_ids = []

    selected_centers = []

    path_rows = []

    candidate_eval_rows = []

    all_ids = candidate_meta[
        "candidate_id"
    ].astype(str).tolist()

    meta_by_id = (
        candidate_meta.set_index(
            "candidate_id"
        )
        .to_dict(
            orient="index"
        )
    )

    for step in range(
        1,
        max_k + 1
    ):

        eligible = []

        for candidate_id in all_ids:

            if candidate_id in selected_ids:
                continue

            center = float(
                meta_by_id[
                    candidate_id
                ][
                    "center_nm"
                ]
            )

            if all(
                abs(
                    center
                    -
                    existing_center
                )
                >=
                min_center_spacing_nm
                -
                1e-9
                for existing_center in selected_centers
            ):
                eligible.append(
                    candidate_id
                )

        if not eligible:
            raise RuntimeError(
                f"Sin candidatos elegibles en step={step}"
            )

        best_candidate = None

        iterator = tqdm(
            eligible,
            desc=(
                f"Forward step {step}/{max_k}"
            ),
            leave=False
        )

        for candidate_id in iterator:

            panel = (
                selected_ids
                +
                [
                    candidate_id
                ]
            )

            best_score, _ = (
                inner_cv_score_raw_panel(
                    development_features,
                    inner_assignment,
                    selected_columns=panel,
                    alpha_grid=SELECTION_ALPHAS
                )
            )

            center = float(
                meta_by_id[
                    candidate_id
                ][
                    "center_nm"
                ]
            )

            candidate_eval_rows.append({
                "step":
                    int(
                        step
                    ),

                "candidate_id":
                    candidate_id,

                "center_nm":
                    center,

                "panel_if_selected":
                    "|".join(
                        panel
                    ),

                "best_alpha":
                    float(
                        best_score[
                            "alpha"
                        ]
                    ),

                "inner_fruit_balanced_mae":
                    float(
                        best_score[
                            "fruit_balanced_mae"
                        ]
                    ),

                "inner_day_balanced_mae":
                    float(
                        best_score[
                            "day_balanced_mae"
                        ]
                    ),

                "inner_mae":
                    float(
                        best_score[
                            "mae"
                        ]
                    ),
            })

            rank = (
                best_score[
                    "fruit_balanced_mae"
                ],
                best_score[
                    "day_balanced_mae"
                ],
                best_score[
                    "mae"
                ],
                center,
            )

            if (
                best_candidate is None
                or
                rank
                <
                best_candidate[
                    "rank"
                ]
            ):

                best_candidate = {
                    "rank":
                        rank,

                    "candidate_id":
                        candidate_id,

                    "center_nm":
                        center,

                    "best_alpha":
                        float(
                            best_score[
                                "alpha"
                            ]
                        ),

                    "inner_fruit_balanced_mae":
                        float(
                            best_score[
                                "fruit_balanced_mae"
                            ]
                        ),

                    "inner_day_balanced_mae":
                        float(
                            best_score[
                                "day_balanced_mae"
                            ]
                        ),

                    "inner_mae":
                        float(
                            best_score[
                                "mae"
                            ]
                        ),
                }

        selected_ids.append(
            best_candidate[
                "candidate_id"
            ]
        )

        selected_centers.append(
            best_candidate[
                "center_nm"
            ]
        )

        path_rows.append({
            "step":
                int(
                    step
                ),

            "selected_candidate_id":
                best_candidate[
                    "candidate_id"
                ],

            "selected_center_nm":
                float(
                    best_candidate[
                        "center_nm"
                    ]
                ),

            "selection_alpha":
                float(
                    best_candidate[
                        "best_alpha"
                    ]
                ),

            "selection_inner_fruit_balanced_mae":
                float(
                    best_candidate[
                        "inner_fruit_balanced_mae"
                    ]
                ),

            "selection_inner_day_balanced_mae":
                float(
                    best_candidate[
                        "inner_day_balanced_mae"
                    ]
                ),

            "panel_after_step":
                "|".join(
                    selected_ids
                ),
        })

    return (
        pd.DataFrame(
            path_rows
        ),
        pd.DataFrame(
            candidate_eval_rows
        )
    )


# 10. Tuning final RAW/SNV + alpha para un panel fijo


In [ ]:
def inner_tune_fixed_panel_standard(
    development_features,
    inner_assignment,
    selected_columns
):

    best = None

    rows = []

    for preprocessing, alpha in itertools.product(
        FINAL_PREPROCESSING,
        FINAL_ALPHAS
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_df = development_features[
                ~development_features[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            val_df = development_features[
                development_features[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            X_train = preprocess_panel(
                train_df[
                    selected_columns
                ].to_numpy(
                    dtype=float
                ),
                preprocessing
            )

            X_val = preprocess_panel(
                val_df[
                    selected_columns
                ].to_numpy(
                    dtype=float
                ),
                preprocessing
            )

            pred = fit_predict_ridge(
                X_train,
                train_df[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                X_val,
                alpha=alpha
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        val_df[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        val_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        pred_df = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            pred_df
        )

        db = day_balanced_mae(
            pred_df
        )

        mae = float(
            mean_absolute_error(
                pred_df[
                    "y_true"
                ],
                pred_df[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "preprocessing":
                preprocessing,

            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            preprocessing,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "preprocessing":
                    preprocessing,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,

                "inner_day_balanced_mae":
                    db,

                "inner_mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 11. Ejecutar selección multivariada y OOF estándar

Se hace **una sola ruta forward hasta 10 canales** por:

\[
outer\ fold 	imes FWHM
\]

Los paneles k=3,5,8,10 son prefijos de esa ruta.

Se crean checkpoints para poder reanudar Colab.


In [ ]:
def checkpoint_stem(
    outer_fold,
    fwhm
):

    return (
        f"outer{int(outer_fold)}_"
        f"fwhm{int(fwhm)}"
    )


standard_pred_parts = []
selection_path_parts = []
candidate_eval_parts = []
final_tuning_parts = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_assignment = inner[
        inner[
            "outer_fold"
        ]
        ==
        outer_fold
    ][
        [
            "fruit_id",
            "inner_fold"
        ]
    ].copy()

    assert set(
        inner_assignment[
            "fruit_id"
        ].astype(int)
    ) == development_fruits

    for fwhm in FWHM_VALUES:

        stem = checkpoint_stem(
            outer_fold,
            fwhm
        )

        path_file = (
            CHECKPOINT_DIR /
            f"{stem}_forward_path.csv"
        )

        eval_file = (
            CHECKPOINT_DIR /
            f"{stem}_candidate_evaluations.csv"
        )

        pred_file = (
            CHECKPOINT_DIR /
            f"{stem}_standard_predictions.csv"
        )

        tuning_file = (
            CHECKPOINT_DIR /
            f"{stem}_final_tuning.csv"
        )

        vf = virtual_features[
            fwhm
        ]

        development_features = vf[
            vf[
                "fruit_id"
            ].isin(
                development_fruits
            )
        ].copy()

        test_features = vf[
            vf[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        candidate_meta = filter_banks[
            fwhm
        ][
            "meta"
        ]

        if (
            RESUME
            and
            path_file.exists()
            and
            pred_file.exists()
            and
            tuning_file.exists()
        ):

            print(
                f"↪ checkpoint outer={outer_fold}, "
                f"FWHM={fwhm:.0f}"
            )

            path_df = pd.read_csv(
                path_file
            )

            pred_df = pd.read_csv(
                pred_file
            )

            tuning_df = pd.read_csv(
                tuning_file
            )

            if eval_file.exists():

                eval_df = pd.read_csv(
                    eval_file
                )

            else:

                eval_df = pd.DataFrame()

        else:

            print(
                "\n",
                "=" * 68,
                sep=""
            )

            print(
                f"OUTER {outer_fold} | "
                f"FWHM {fwhm:.0f} nm"
            )

            print(
                "=" * 68
            )

            path_df, eval_df = (
                forward_select_virtual_filters(
                    development_features=
                        development_features,

                    inner_assignment=
                        inner_assignment,

                    candidate_meta=
                        candidate_meta,

                    max_k=
                        MAX_K,

                    min_center_spacing_nm=
                        float(
                            fwhm
                        ),
                )
            )

            path_df[
                "outer_fold"
            ] = int(
                outer_fold
            )

            path_df[
                "fwhm_nm"
            ] = float(
                fwhm
            )

            eval_df[
                "outer_fold"
            ] = int(
                outer_fold
            )

            eval_df[
                "fwhm_nm"
            ] = float(
                fwhm
            )

            pred_list = []

            tuning_list = []

            ordered_ids = path_df[
                "selected_candidate_id"
            ].astype(
                str
            ).tolist()

            for k in K_VALUES:

                panel = ordered_ids[
                    :k
                ]

                best, tuning = (
                    inner_tune_fixed_panel_standard(
                        development_features,
                        inner_assignment,
                        panel
                    )
                )

                X_train = preprocess_panel(
                    development_features[
                        panel
                    ].to_numpy(
                        dtype=float
                    ),
                    best[
                        "preprocessing"
                    ]
                )

                X_test = preprocess_panel(
                    test_features[
                        panel
                    ].to_numpy(
                        dtype=float
                    ),
                    best[
                        "preprocessing"
                    ]
                )

                pred = fit_predict_ridge(
                    X_train,
                    development_features[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    X_test,
                    alpha=best[
                        "alpha"
                    ]
                )

                centers = (
                    candidate_meta.set_index(
                        "candidate_id"
                    )
                    .loc[
                        panel,
                        "center_nm"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )

                pred_list.append(
                    pd.DataFrame({
                        "camera_type":
                            CAMERA,

                        "protocol":
                            "standard_fruit_disjoint",

                        "outer_fold":
                            int(
                                outer_fold
                            ),

                        "fwhm_nm":
                            float(
                                fwhm
                            ),

                        "k":
                            int(
                                k
                            ),

                        "model":
                            (
                                f"VirtualRidge_"
                                f"FWHM{int(fwhm)}_"
                                f"k{k}"
                            ),

                        "preprocessing":
                            best[
                                "preprocessing"
                            ],

                        "alpha":
                            float(
                                best[
                                    "alpha"
                                ]
                            ),

                        "panel_ids":
                            "|".join(
                                panel
                            ),

                        "panel_centers_nm":
                            "|".join(
                                [
                                    f"{x:.1f}"
                                    for x in centers
                                ]
                            ),

                        "fruit_id":
                            test_features[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "storage_days":
                            test_features[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            test_features[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

                tuning[
                    "outer_fold"
                ] = int(
                    outer_fold
                )

                tuning[
                    "fwhm_nm"
                ] = float(
                    fwhm
                )

                tuning[
                    "k"
                ] = int(
                    k
                )

                tuning[
                    "panel_ids"
                ] = "|".join(
                    panel
                )

                tuning_list.append(
                    tuning
                )

            pred_df = pd.concat(
                pred_list,
                ignore_index=True
            )

            tuning_df = pd.concat(
                tuning_list,
                ignore_index=True
            )

            path_df.to_csv(
                path_file,
                index=False
            )

            eval_df.to_csv(
                eval_file,
                index=False
            )

            pred_df.to_csv(
                pred_file,
                index=False
            )

            tuning_df.to_csv(
                tuning_file,
                index=False
            )

        standard_pred_parts.append(
            pred_df
        )

        selection_path_parts.append(
            path_df
        )

        if len(
            eval_df
        ) > 0:

            candidate_eval_parts.append(
                eval_df
            )

        final_tuning_parts.append(
            tuning_df
        )


standard_oof = pd.concat(
    standard_pred_parts,
    ignore_index=True
)

selection_paths = pd.concat(
    selection_path_parts,
    ignore_index=True
)

candidate_evaluations = (
    pd.concat(
        candidate_eval_parts,
        ignore_index=True
    )
    if candidate_eval_parts
    else
    pd.DataFrame()
)

final_tuning_standard = pd.concat(
    final_tuning_parts,
    ignore_index=True
)

for fwhm in FWHM_VALUES:

    for k in K_VALUES:

        g = standard_oof[
            (
                standard_oof[
                    "fwhm_nm"
                ]
                ==
                fwhm
            )
            &
            (
                standard_oof[
                    "k"
                ]
                ==
                k
            )
        ]

        assert len(
            g
        ) == 199

        assert g[
            "fruit_id"
        ].nunique() == 40

print("✅ OOF estándar multivariado completo.")


## 12. Métricas OOF estándar


In [ ]:
standard_metric_rows = []

for (
    fwhm,
    k
), g in standard_oof.groupby(
    [
        "fwhm_nm",
        "k"
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            CAMERA,

        "protocol":
            "standard_fruit_disjoint",

        "fwhm_nm":
            float(
                fwhm
            ),

        "k":
            int(
                k
            ),

        "model":
            (
                f"VirtualRidge_"
                f"FWHM{int(fwhm)}_"
                f"k{int(k)}"
            ),
    })

    standard_metric_rows.append(
        m
    )

standard_metrics = pd.DataFrame(
    standard_metric_rows
).sort_values(
    [
        "fruit_balanced_mae",
        "k",
        "fwhm_nm"
    ]
)

display(
    standard_metrics
)

standard_metrics.to_csv(
    TABLES /
    "02_virtual_sensor_standard_metrics.csv",
    index=False
)

selection_paths.to_csv(
    TABLES /
    "03_forward_selection_paths.csv",
    index=False
)

if len(
    candidate_evaluations
) > 0:

    candidate_evaluations.to_csv(
        TABLES /
        "04_forward_candidate_evaluations.csv",
        index=False
    )

final_tuning_standard.to_csv(
    TABLES /
    "05_standard_final_tuning.csv",
    index=False
)


# 13. Referencias full-spectrum y notebook 05


In [ ]:
full_standard = baseline_oof[
    (
        baseline_oof[
            "camera_type"
        ]
        ==
        CAMERA
    )
    &
    (
        baseline_oof[
            "model"
        ]
        ==
        "Ridge"
    )
].copy()

assert len(
    full_standard
) == 199

full_standard[
    "model"
] = "Full249_Ridge_standard"

full_standard_metrics = metric_bundle(
    full_standard
)

dummy_standard = baseline_oof[
    (
        baseline_oof[
            "camera_type"
        ]
        ==
        CAMERA
    )
    &
    (
        baseline_oof[
            "model"
        ]
        ==
        "DummyMedian"
    )
].copy()

dummy_standard_metrics = metric_bundle(
    dummy_standard
)

full_session = full_session_oof_raw[
    full_session_oof_raw[
        "camera_type"
    ]
    ==
    CAMERA
].copy()

assert len(
    full_session
) == 199

full_session[
    "model"
] = "Full249_Ridge_session_excluded"

full_session_metrics = metric_bundle(
    full_session
)

previous05_standard = (
    pd.read_csv(
        OPTIONAL_05_STANDARD
    )
    if OPTIONAL_05_STANDARD.exists()
    else None
)

previous05_session = (
    pd.read_csv(
        OPTIONAL_05_SESSION
    )
    if OPTIONAL_05_SESSION.exists()
    else None
)

print(
    "Full249 standard FB-MAE:",
    full_standard_metrics[
        "fruit_balanced_mae"
    ]
)

print(
    "Full249 session-excluded FB-MAE:",
    full_session_metrics[
        "fruit_balanced_mae"
    ]
)

print(
    "Notebook 05 disponible:",
    previous05_standard is not None
)


# 14. Construir null correcto session-excluded


In [ ]:
session_null_parts = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_df = data[
        ~data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    test_df = data[
        data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    for target_day in sorted(
        test_df[
            "storage_days_model"
        ].unique()
    ):

        train_excluded = development_df[
            development_df[
                "storage_days_model"
            ]
            !=
            target_day
        ]

        test_day = test_df[
            test_df[
                "storage_days_model"
            ]
            ==
            target_day
        ]

        null_value = float(
            train_excluded[
                "storage_days_model"
            ].mean()
        )

        session_null_parts.append(
            pd.DataFrame({
                "camera_type":
                    CAMERA,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "model":
                    "SessionExcluded_DummyMean",

                "fruit_id":
                    test_day[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    test_day[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    test_day[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.repeat(
                        null_value,
                        len(
                            test_day
                        )
                    ),
            })
        )

session_null_oof = pd.concat(
    session_null_parts,
    ignore_index=True
)

assert len(
    session_null_oof
) == 199

session_null_metrics = metric_bundle(
    session_null_oof
)

print(
    "Session-excluded null FB-MAE:",
    session_null_metrics[
        "fruit_balanced_mae"
    ]
)


# 15. Tuning session-excluded para panel fijo


In [ ]:
def inner_tune_session_excluded_fixed_panel(
    development_features,
    inner_assignment,
    panel_columns
):

    best = None

    rows = []

    for preprocessing, alpha in itertools.product(
        FINAL_PREPROCESSING,
        FINAL_ALPHAS
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            inner_train = development_features[
                ~development_features[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            inner_val = development_features[
                development_features[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            for target_day in sorted(
                inner_val[
                    "storage_days_model"
                ].unique()
            ):

                train_excluded = inner_train[
                    inner_train[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ].copy()

                val_day = inner_val[
                    inner_val[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ].copy()

                X_train = preprocess_panel(
                    train_excluded[
                        panel_columns
                    ].to_numpy(
                        dtype=float
                    ),
                    preprocessing
                )

                X_val = preprocess_panel(
                    val_day[
                        panel_columns
                    ].to_numpy(
                        dtype=float
                    ),
                    preprocessing
                )

                pred = fit_predict_ridge(
                    X_train,
                    train_excluded[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    X_val,
                    alpha=alpha
                )

                pred_parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            val_day[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            val_day[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

        pred_df = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            pred_df
        )

        db = day_balanced_mae(
            pred_df
        )

        mae = float(
            mean_absolute_error(
                pred_df[
                    "y_true"
                ],
                pred_df[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "preprocessing":
                preprocessing,

            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            preprocessing,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "preprocessing":
                    preprocessing,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,

                "inner_day_balanced_mae":
                    db,

                "inner_mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 16. OOF session-excluded con los paneles virtuales fijos


In [ ]:
session_pred_parts = []
session_tuning_parts = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_assignment = inner[
        inner[
            "outer_fold"
        ]
        ==
        outer_fold
    ][
        [
            "fruit_id",
            "inner_fold"
        ]
    ].copy()

    for fwhm in FWHM_VALUES:

        vf = virtual_features[
            fwhm
        ]

        development_features = vf[
            vf[
                "fruit_id"
            ].isin(
                development_fruits
            )
        ].copy()

        test_features = vf[
            vf[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        path_df = selection_paths[
            (
                selection_paths[
                    "outer_fold"
                ]
                ==
                outer_fold
            )
            &
            (
                selection_paths[
                    "fwhm_nm"
                ]
                ==
                fwhm
            )
        ].sort_values(
            "step"
        )

        ordered_ids = path_df[
            "selected_candidate_id"
        ].astype(
            str
        ).tolist()

        for k in K_VALUES:

            panel = ordered_ids[
                :k
            ]

            best, tuning_df = (
                inner_tune_session_excluded_fixed_panel(
                    development_features,
                    inner_assignment,
                    panel
                )
            )

            panel_centers = (
                filter_banks[
                    fwhm
                ][
                    "meta"
                ]
                .set_index(
                    "candidate_id"
                )
                .loc[
                    panel,
                    "center_nm"
                ]
                .to_numpy(
                    dtype=float
                )
            )

            for target_day in sorted(
                test_features[
                    "storage_days_model"
                ].unique()
            ):

                train_excluded = development_features[
                    development_features[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ].copy()

                test_day = test_features[
                    test_features[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ].copy()

                X_train = preprocess_panel(
                    train_excluded[
                        panel
                    ].to_numpy(
                        dtype=float
                    ),
                    best[
                        "preprocessing"
                    ]
                )

                X_test = preprocess_panel(
                    test_day[
                        panel
                    ].to_numpy(
                        dtype=float
                    ),
                    best[
                        "preprocessing"
                    ]
                )

                pred = fit_predict_ridge(
                    X_train,
                    train_excluded[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    X_test,
                    alpha=best[
                        "alpha"
                    ]
                )

                session_pred_parts.append(
                    pd.DataFrame({
                        "camera_type":
                            CAMERA,

                        "protocol":
                            "session_excluded_fixed_virtual_panel",

                        "outer_fold":
                            int(
                                outer_fold
                            ),

                        "fwhm_nm":
                            float(
                                fwhm
                            ),

                        "k":
                            int(
                                k
                            ),

                        "model":
                            (
                                f"VirtualRidge_"
                                f"FWHM{int(fwhm)}_"
                                f"k{k}_session_excluded"
                            ),

                        "preprocessing":
                            best[
                                "preprocessing"
                            ],

                        "alpha":
                            float(
                                best[
                                    "alpha"
                                ]
                            ),

                        "panel_ids":
                            "|".join(
                                panel
                            ),

                        "panel_centers_nm":
                            "|".join(
                                [
                                    f"{x:.1f}"
                                    for x in panel_centers
                                ]
                            ),

                        "fruit_id":
                            test_day[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "storage_days":
                            test_day[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            test_day[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

            tuning_df[
                "outer_fold"
            ] = int(
                outer_fold
            )

            tuning_df[
                "fwhm_nm"
            ] = float(
                fwhm
            )

            tuning_df[
                "k"
            ] = int(
                k
            )

            tuning_df[
                "panel_ids"
            ] = "|".join(
                panel
            )

            session_tuning_parts.append(
                tuning_df
            )

session_oof = pd.concat(
    session_pred_parts,
    ignore_index=True
)

session_tuning = pd.concat(
    session_tuning_parts,
    ignore_index=True
)

for fwhm in FWHM_VALUES:

    for k in K_VALUES:

        g = session_oof[
            (
                session_oof[
                    "fwhm_nm"
                ]
                ==
                fwhm
            )
            &
            (
                session_oof[
                    "k"
                ]
                ==
                k
            )
        ]

        assert len(
            g
        ) == 199

        assert g[
            "fruit_id"
        ].nunique() == 40

print("✅ OOF session-excluded virtual completo.")


## 17. Métricas session-excluded


In [ ]:
session_metric_rows = []

for (
    fwhm,
    k
), g in session_oof.groupby(
    [
        "fwhm_nm",
        "k"
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            CAMERA,

        "protocol":
            "session_excluded_fixed_virtual_panel",

        "fwhm_nm":
            float(
                fwhm
            ),

        "k":
            int(
                k
            ),

        "model":
            (
                f"VirtualRidge_"
                f"FWHM{int(fwhm)}_"
                f"k{int(k)}_session_excluded"
            ),
    })

    session_metric_rows.append(
        m
    )

session_metrics = pd.DataFrame(
    session_metric_rows
).sort_values(
    [
        "fruit_balanced_mae",
        "k",
        "fwhm_nm"
    ]
)

display(
    session_metrics
)

session_metrics.to_csv(
    TABLES /
    "06_virtual_sensor_session_excluded_metrics.csv",
    index=False
)

session_tuning.to_csv(
    TABLES /
    "07_session_excluded_final_tuning.csv",
    index=False
)


# 18. Comparación maestra: full vs virtual multivariado


In [ ]:
comparison_rows = []

r = full_standard_metrics.copy()

r.update({
    "protocol":
        "standard",

    "sensor":
        "Full249",

    "fwhm_nm":
        np.nan,

    "k":
        249,
})

comparison_rows.append(
    r
)

r = dummy_standard_metrics.copy()

r.update({
    "protocol":
        "standard",

    "sensor":
        "DummyMedian",

    "fwhm_nm":
        np.nan,

    "k":
        0,
})

comparison_rows.append(
    r
)

for _, row in standard_metrics.iterrows():

    z = row.to_dict()

    z[
        "protocol"
    ] = "standard"

    z[
        "sensor"
    ] = (
        f"Virtual_FWHM"
        f"{int(row['fwhm_nm'])}_"
        f"k{int(row['k'])}"
    )

    comparison_rows.append(
        z
    )

r = full_session_metrics.copy()

r.update({
    "protocol":
        "session_excluded",

    "sensor":
        "Full249",

    "fwhm_nm":
        np.nan,

    "k":
        249,
})

comparison_rows.append(
    r
)

r = session_null_metrics.copy()

r.update({
    "protocol":
        "session_excluded",

    "sensor":
        "SessionExcluded_DummyMean",

    "fwhm_nm":
        np.nan,

    "k":
        0,
})

comparison_rows.append(
    r
)

for _, row in session_metrics.iterrows():

    z = row.to_dict()

    z[
        "protocol"
    ] = "session_excluded"

    z[
        "sensor"
    ] = (
        f"Virtual_FWHM"
        f"{int(row['fwhm_nm'])}_"
        f"k{int(row['k'])}"
    )

    comparison_rows.append(
        z
    )

comparison_metrics = pd.DataFrame(
    comparison_rows
)

display(
    comparison_metrics.sort_values(
        [
            "protocol",
            "fruit_balanced_mae"
        ]
    )
)

comparison_metrics.to_csv(
    TABLES /
    "08_full_vs_virtual_comparison_metrics.csv",
    index=False
)


# 19. Bootstrap pareado por fruto


In [ ]:
def per_fruit_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return (
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
    )


def paired_bootstrap(
    a,
    b,
    label,
    n_boot=5000,
    seed=20260818
):

    ids = sorted(
        set(
            a.index
        )
        &
        set(
            b.index
        )
    )

    av = a.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    bv = b.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    diff = (
        av
        -
        bv
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(
                diff
            ),
            size=len(
                diff
            )
        )

        boot[
            i
        ] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            label,

        "n_fruits":
            int(
                len(
                    diff
                )
            ),

        "mean_delta_mae_days":
            float(
                diff.mean()
            ),

        "median_delta_mae_days":
            float(
                np.median(
                    diff
                )
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_first_better":
            float(
                np.mean(
                    diff
                    <
                    0
                )
            ),
    }


full_std_pf = per_fruit_mae(
    full_standard
)

full_sess_pf = per_fruit_mae(
    full_session
)

null_sess_pf = per_fruit_mae(
    session_null_oof
)

bootstrap_rows = []

counter = 0

for (
    fwhm,
    k
), g in standard_oof.groupby(
    [
        "fwhm_nm",
        "k"
    ]
):

    virtual_pf = per_fruit_mae(
        g
    )

    row = paired_bootstrap(
        virtual_pf,
        full_std_pf,
        (
            f"standard virtual FWHM={fwhm:.0f} "
            f"k={int(k)} minus Full249"
        ),
        n_boot=BOOTSTRAP_REPS,
        seed=RANDOM_SEED + counter
    )

    row.update({
        "protocol":
            "standard_vs_full",

        "fwhm_nm":
            float(
                fwhm
            ),

        "k":
            int(
                k
            ),
    })

    bootstrap_rows.append(
        row
    )

    counter += 1


for (
    fwhm,
    k
), g in session_oof.groupby(
    [
        "fwhm_nm",
        "k"
    ]
):

    virtual_pf = per_fruit_mae(
        g
    )

    row = paired_bootstrap(
        virtual_pf,
        full_sess_pf,
        (
            f"session virtual FWHM={fwhm:.0f} "
            f"k={int(k)} minus Full249 session"
        ),
        n_boot=BOOTSTRAP_REPS,
        seed=RANDOM_SEED + 100 + counter
    )

    row.update({
        "protocol":
            "session_vs_full",

        "fwhm_nm":
            float(
                fwhm
            ),

        "k":
            int(
                k
            ),
    })

    bootstrap_rows.append(
        row
    )

    row = paired_bootstrap(
        virtual_pf,
        null_sess_pf,
        (
            f"session virtual FWHM={fwhm:.0f} "
            f"k={int(k)} minus correct null"
        ),
        n_boot=BOOTSTRAP_REPS,
        seed=RANDOM_SEED + 200 + counter
    )

    row.update({
        "protocol":
            "session_vs_null",

        "fwhm_nm":
            float(
                fwhm
            ),

        "k":
            int(
                k
            ),
    })

    bootstrap_rows.append(
        row
    )

    counter += 1


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES /
    "09_paired_fruit_bootstrap_virtual_sensor.csv",
    index=False
)


# 20. Retención de desempeño estándar y session-excluded


In [ ]:
full_std_mae = float(
    full_standard_metrics[
        "fruit_balanced_mae"
    ]
)

full_sess_mae = float(
    full_session_metrics[
        "fruit_balanced_mae"
    ]
)

null_sess_mae = float(
    session_null_metrics[
        "fruit_balanced_mae"
    ]
)

retention_rows = []

for _, row in standard_metrics.iterrows():

    retention_rows.append({
        "protocol":
            "standard",

        "fwhm_nm":
            float(
                row[
                    "fwhm_nm"
                ]
            ),

        "k":
            int(
                row[
                    "k"
                ]
            ),

        "virtual_fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "reference_full249_mae":
            full_std_mae,

        "delta_vs_full_days":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                -
                full_std_mae
            ),

        "mae_ratio_virtual_over_full":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                /
                full_std_mae
            ),

        "improvement_over_correct_null_days":
            np.nan,
    })

for _, row in session_metrics.iterrows():

    retention_rows.append({
        "protocol":
            "session_excluded",

        "fwhm_nm":
            float(
                row[
                    "fwhm_nm"
                ]
            ),

        "k":
            int(
                row[
                    "k"
                ]
            ),

        "virtual_fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "reference_full249_mae":
            full_sess_mae,

        "delta_vs_full_days":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                -
                full_sess_mae
            ),

        "mae_ratio_virtual_over_full":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                /
                full_sess_mae
            ),

        "improvement_over_correct_null_days":
            float(
                null_sess_mae
                -
                row[
                    "fruit_balanced_mae"
                ]
            ),
    })

retention = pd.DataFrame(
    retention_rows
)

display(
    retention.sort_values(
        [
            "protocol",
            "virtual_fruit_balanced_mae"
        ]
    )
)

retention.to_csv(
    TABLES /
    "10_virtual_sensor_performance_retention.csv",
    index=False
)


# 21. Figura principal — MAE vs número de canales


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 5.3)
)

for fwhm in FWHM_VALUES:

    g = standard_metrics[
        standard_metrics[
            "fwhm_nm"
        ]
        ==
        fwhm
    ].sort_values(
        "k"
    )

    ax.plot(
        g[
            "k"
        ],
        g[
            "fruit_balanced_mae"
        ],
        marker="o",
        label=(
            f"Estándar FWHM {int(fwhm)} nm"
        )
    )

for fwhm in FWHM_VALUES:

    g = session_metrics[
        session_metrics[
            "fwhm_nm"
        ]
        ==
        fwhm
    ].sort_values(
        "k"
    )

    ax.plot(
        g[
            "k"
        ],
        g[
            "fruit_balanced_mae"
        ],
        marker="s",
        linestyle="--",
        label=(
            f"Session-excl. FWHM {int(fwhm)} nm"
        )
    )

ax.axhline(
    full_std_mae,
    linestyle="-.",
    label="Full249 estándar"
)

ax.axhline(
    full_sess_mae,
    linestyle=":",
    label="Full249 session-excluded"
)

ax.axhline(
    null_sess_mae,
    linestyle="--",
    alpha=0.7,
    label="Null session-excluded"
)

ax.set_xlabel(
    "Número de canales virtuales"
)

ax.set_ylabel(
    "MAE fruit-balanced (días)"
)

ax.set_title(
    "VIS_COR — diseño multivariado de sensor virtual"
)

ax.legend(
    fontsize=8,
    ncol=2
)

ax.grid(
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "02_virtual_sensor_mae_vs_channels.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 22. Comparación opcional con selector univariado del notebook 05


In [ ]:
previous05_summary_rows = []

if previous05_standard is not None:

    for k, g in previous05_standard.groupby(
        "k"
    ):

        m = metric_bundle(
            g
        )

        previous05_summary_rows.append({
            "source":
                "Notebook05_univariate",

            "protocol":
                "standard",

            "k":
                int(
                    k
                ),

            "fruit_balanced_mae":
                float(
                    m[
                        "fruit_balanced_mae"
                    ]
                ),
        })

if previous05_session is not None:

    for k, g in previous05_session.groupby(
        "k"
    ):

        m = metric_bundle(
            g
        )

        previous05_summary_rows.append({
            "source":
                "Notebook05_univariate",

            "protocol":
                "session_excluded",

            "k":
                int(
                    k
                ),

            "fruit_balanced_mae":
                float(
                    m[
                        "fruit_balanced_mae"
                    ]
                ),
        })

previous05_summary = pd.DataFrame(
    previous05_summary_rows
)

if len(
    previous05_summary
) > 0:

    best_virtual_by_k = (
        standard_metrics.sort_values(
            [
                "k",
                "fruit_balanced_mae"
            ]
        )
        .groupby(
            "k"
        )
        .head(
            1
        )[
            [
                "k",
                "fwhm_nm",
                "fruit_balanced_mae"
            ]
        ]
        .rename(
            columns={
                "fruit_balanced_mae":
                    "best_virtual_standard_mae"
            }
        )
    )

    old_standard = previous05_summary[
        previous05_summary[
            "protocol"
        ]
        ==
        "standard"
    ][
        [
            "k",
            "fruit_balanced_mae"
        ]
    ].rename(
        columns={
            "fruit_balanced_mae":
                "notebook05_standard_mae"
        }
    )

    comparison05 = (
        best_virtual_by_k.merge(
            old_standard,
            on="k",
            how="left"
        )
    )

    comparison05[
        "improvement_virtual_over_notebook05_days"
    ] = (
        comparison05[
            "notebook05_standard_mae"
        ]
        -
        comparison05[
            "best_virtual_standard_mae"
        ]
    )

    display(
        comparison05
    )

    comparison05.to_csv(
        TABLES /
        "11_virtual_vs_notebook05_univariate.csv",
        index=False
    )

else:

    comparison05 = pd.DataFrame()

    print(
        "Notebook 05 no encontrado en Drive; "
        "se omite comparación opcional."
    )


# 23. Estabilidad de centros seleccionados entre outer folds


In [ ]:
selected_center_rows = []

for _, row in selection_paths.iterrows():

    selected_center_rows.append({
        "outer_fold":
            int(
                row[
                    "outer_fold"
                ]
            ),

        "fwhm_nm":
            float(
                row[
                    "fwhm_nm"
                ]
            ),

        "step":
            int(
                row[
                    "step"
                ]
            ),

        "candidate_id":
            str(
                row[
                    "selected_candidate_id"
                ]
            ),

        "center_nm":
            float(
                row[
                    "selected_center_nm"
                ]
            ),
    })

selected_centers = pd.DataFrame(
    selected_center_rows
)

selection_frequency = (
    selected_centers.groupby(
        [
            "fwhm_nm",
            "candidate_id",
            "center_nm"
        ]
    )
    .agg(
        n_outer_folds_selected=(
            "outer_fold",
            "nunique"
        ),
        mean_selection_step=(
            "step",
            "mean"
        ),
        min_selection_step=(
            "step",
            "min"
        ),
    )
    .reset_index()
    .sort_values(
        [
            "fwhm_nm",
            "n_outer_folds_selected",
            "mean_selection_step"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)

display(
    selection_frequency
)

selection_frequency.to_csv(
    TABLES /
    "12_virtual_filter_selection_stability.csv",
    index=False
)


## 24. Figura — rutas de selección por outer fold


In [ ]:
for fwhm in FWHM_VALUES:

    g = selected_centers[
        selected_centers[
            "fwhm_nm"
        ]
        ==
        fwhm
    ]

    fig, ax = plt.subplots(
        figsize=(10, 4.8)
    )

    for fold in range(
        OUTER_FOLDS
    ):

        f = g[
            g[
                "outer_fold"
            ]
            ==
            fold
        ].sort_values(
            "step"
        )

        ax.scatter(
            f[
                "center_nm"
            ],
            np.repeat(
                fold,
                len(
                    f
                )
            ),
            s=50
        )

        for _, row in f.iterrows():

            ax.text(
                row[
                    "center_nm"
                ],
                fold
                +
                0.08,
                str(
                    int(
                        row[
                            "step"
                        ]
                    )
                ),
                fontsize=7,
                ha="center"
            )

    ax.set_xlabel(
        "Centro del filtro (nm)"
    )

    ax.set_ylabel(
        "Outer fold"
    )

    ax.set_yticks(
        range(
            OUTER_FOLDS
        )
    )

    ax.set_title(
        f"FWHM {int(fwhm)} nm — ruta forward por fold\n"
        "número = orden de entrada"
    )

    ax.grid(
        axis="x",
        alpha=0.15
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"03_forward_selection_paths_fwhm{int(fwhm)}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 25. Error y bias por día


In [ ]:
day_rows = []

for protocol_name, pred_df in [
    (
        "standard",
        standard_oof
    ),
    (
        "session_excluded",
        session_oof
    ),
]:

    for (
        fwhm,
        k,
        day
    ), g in pred_df.groupby(
        [
            "fwhm_nm",
            "k",
            "storage_days"
        ]
    ):

        err = (
            g[
                "y_pred"
            ].to_numpy(
                dtype=float
            )
            -
            g[
                "y_true"
            ].to_numpy(
                dtype=float
            )
        )

        day_rows.append({
            "protocol":
                protocol_name,

            "fwhm_nm":
                float(
                    fwhm
                ),

            "k":
                int(
                    k
                ),

            "storage_days":
                int(
                    day
                ),

            "n":
                int(
                    len(
                        g
                    )
                ),

            "mae":
                float(
                    np.mean(
                        np.abs(
                            err
                        )
                    )
                ),

            "bias":
                float(
                    np.mean(
                        err
                    )
                ),
        })

day_metrics = pd.DataFrame(
    day_rows
)

day_metrics.to_csv(
    TABLES /
    "13_virtual_sensor_error_bias_by_day.csv",
    index=False
)

display(
    day_metrics.head(
        30
    )
)


# 26. Identificar configuraciones Pareto / pantalla de ingeniería

Se reportan dos cosas:

1. **Mejor desempeño virtual**, independientemente de número de canales.
2. **Panel más pequeño dentro de +0.10 días del Full249**, si existe.

La pantalla `+0.10 d` es descriptiva; no es prueba formal de equivalencia.


In [ ]:
standard_ranked = standard_metrics.sort_values(
    [
        "fruit_balanced_mae",
        "k",
        "fwhm_nm"
    ]
).reset_index(
    drop=True
)

best_virtual_standard = standard_ranked.iloc[
    0
]

standard_screen = standard_metrics.copy()

standard_screen[
    "delta_vs_full249_days"
] = (
    standard_screen[
        "fruit_balanced_mae"
    ]
    -
    full_std_mae
)

standard_screen[
    "within_engineering_screen"
] = (
    standard_screen[
        "delta_vs_full249_days"
    ]
    <=
    ENGINEERING_DELTA_DAYS
)

eligible = standard_screen[
    standard_screen[
        "within_engineering_screen"
    ]
].sort_values(
    [
        "k",
        "fruit_balanced_mae"
    ]
)

if len(
    eligible
) > 0:

    smallest_engineering_panel = (
        eligible.iloc[
            0
        ]
    )

else:

    smallest_engineering_panel = None

display(
    standard_screen.sort_values(
        [
            "k",
            "fwhm_nm"
        ]
    )
)

standard_screen.to_csv(
    TABLES /
    "14_virtual_sensor_engineering_screen.csv",
    index=False
)

print(
    "\nMejor panel virtual estándar:"
)

display(
    pd.DataFrame(
        [
            best_virtual_standard
        ]
    )
)

if smallest_engineering_panel is None:

    print(
        "Ningún panel quedó dentro de +0.10 d del Full249."
    )

else:

    print(
        "\nPanel más pequeño dentro de +0.10 d:"
    )

    display(
        pd.DataFrame(
            [
                smallest_engineering_panel
            ]
        )
    )


# 27. Panel candidato final para trabajo futuro

Después de terminar la estimación OOF, se diseña un panel candidato usando **los 40 frutos completos**.

Esto NO genera una nueva estimación de desempeño.

Se usa:

- el FWHM del mejor panel virtual estándar;
- el `k` del mejor panel virtual estándar;
- los 5 outer folds congelados como CV de diseño.

El panel resultante se considera únicamente una propuesta de sensor para validación futura.


In [ ]:
best_fwhm = float(
    best_virtual_standard[
        "fwhm_nm"
    ]
)

best_k = int(
    best_virtual_standard[
        "k"
    ]
)

all_features = virtual_features[
    best_fwhm
].copy()

candidate_meta = filter_banks[
    best_fwhm
][
    "meta"
]

# Reutilizar los outer folds como folds de diseño sobre los 40 frutos.
design_assignment = outer[
    [
        "fruit_id",
        "outer_fold"
    ]
].rename(
    columns={
        "outer_fold":
            "inner_fold"
    }
).copy()

# La función espera 4 folds; hacemos una versión general de scoring.
def cv_score_raw_panel_general(
    features_df,
    fold_assignment,
    selected_columns,
    alpha_grid
):

    fold_values = sorted(
        fold_assignment[
            "inner_fold"
        ].unique()
    )

    best = None

    for alpha in alpha_grid:

        parts = []

        for fold_value in fold_values:

            val_fruits = set(
                fold_assignment.loc[
                    fold_assignment[
                        "inner_fold"
                    ]
                    ==
                    fold_value,
                    "fruit_id"
                ].astype(int)
            )

            tr = features_df[
                ~features_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            va = features_df[
                features_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            pred = fit_predict_ridge(
                tr[
                    selected_columns
                ].to_numpy(
                    dtype=float
                ),
                tr[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                va[
                    selected_columns
                ].to_numpy(
                    dtype=float
                ),
                alpha=alpha
            )

            parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            parts,
            ignore_index=True
        )

        rank = (
            fruit_balanced_mae(
                p
            ),
            day_balanced_mae(
                p
            ),
            float(
                mean_absolute_error(
                    p[
                        "y_true"
                    ],
                    p[
                        "y_pred"
                    ]
                )
            ),
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),
            }

    return best


def forward_design_general(
    features_df,
    fold_assignment,
    candidate_meta,
    max_k,
    min_spacing_nm
):

    selected = []

    centers = []

    rows = []

    meta_by_id = (
        candidate_meta.set_index(
            "candidate_id"
        )
        .to_dict(
            orient="index"
        )
    )

    all_ids = candidate_meta[
        "candidate_id"
    ].astype(
        str
    ).tolist()

    for step in range(
        1,
        max_k + 1
    ):

        best_candidate = None

        eligible = [
            cid
            for cid in all_ids
            if (
                cid not in selected
                and
                all(
                    abs(
                        float(
                            meta_by_id[
                                cid
                            ][
                                "center_nm"
                            ]
                        )
                        -
                        c
                    )
                    >=
                    min_spacing_nm
                    -
                    1e-9
                    for c in centers
                )
            )
        ]

        for cid in tqdm(
            eligible,
            desc=(
                f"Final design step {step}/{max_k}"
            ),
            leave=False
        ):

            panel = (
                selected
                +
                [
                    cid
                ]
            )

            score = (
                cv_score_raw_panel_general(
                    features_df,
                    fold_assignment,
                    panel,
                    SELECTION_ALPHAS
                )
            )

            center = float(
                meta_by_id[
                    cid
                ][
                    "center_nm"
                ]
            )

            rank = (
                score[
                    "rank"
                ][
                    0
                ],
                score[
                    "rank"
                ][
                    1
                ],
                score[
                    "rank"
                ][
                    2
                ],
                center,
            )

            if (
                best_candidate is None
                or
                rank
                <
                best_candidate[
                    "rank"
                ]
            ):

                best_candidate = {
                    "rank":
                        rank,

                    "candidate_id":
                        cid,

                    "center_nm":
                        center,

                    "alpha":
                        float(
                            score[
                                "alpha"
                            ]
                        ),
                }

        selected.append(
            best_candidate[
                "candidate_id"
            ]
        )

        centers.append(
            best_candidate[
                "center_nm"
            ]
        )

        rows.append({
            "step":
                int(
                    step
                ),

            "candidate_id":
                best_candidate[
                    "candidate_id"
                ],

            "center_nm":
                float(
                    best_candidate[
                        "center_nm"
                    ]
                ),

            "selection_alpha":
                float(
                    best_candidate[
                        "alpha"
                    ]
                ),

            "cv_fruit_balanced_mae":
                float(
                    best_candidate[
                        "rank"
                    ][
                        0
                    ]
                ),
        })

    return pd.DataFrame(
        rows
    )


final_candidate_panel = forward_design_general(
    all_features,
    design_assignment,
    candidate_meta,
    max_k=best_k,
    min_spacing_nm=best_fwhm
)

final_candidate_panel[
    "fwhm_nm"
] = best_fwhm

final_candidate_panel[
    "k_total"
] = best_k

display(
    final_candidate_panel
)

final_candidate_panel.to_csv(
    TABLES /
    "15_final_candidate_virtual_sensor_panel.csv",
    index=False
)

print(
    "⚠️ Este panel usa los 40 frutos para diseño. "
    "No tiene desempeño independiente."
)


# 28. Visualizar respuesta espectral del panel candidato final


In [ ]:
meta = filter_banks[
    best_fwhm
][
    "meta"
].set_index(
    "candidate_id"
)

W = filter_banks[
    best_fwhm
][
    "weights"
]

all_meta = filter_banks[
    best_fwhm
][
    "meta"
]

id_to_index = {
    cid:
        idx
    for idx, cid in enumerate(
        all_meta[
            "candidate_id"
        ]
    )
}

fig, ax = plt.subplots(
    figsize=(10, 5)
)

for _, row in final_candidate_panel.iterrows():

    cid = row[
        "candidate_id"
    ]

    idx = id_to_index[
        cid
    ]

    ax.plot(
        wavelengths,
        W[
            :,
            idx
        ],
        label=(
            f"{row['center_nm']:.0f} nm"
        )
    )

ax.set_xlabel(
    "Longitud de onda (nm)"
)

ax.set_ylabel(
    "Respuesta relativa"
)

ax.set_title(
    (
        f"Panel candidato final — "
        f"{best_k} canales, "
        f"FWHM {best_fwhm:.0f} nm"
    )
)

ax.legend(
    ncol=2
)

ax.grid(
    alpha=0.15
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "04_final_candidate_virtual_sensor_response.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 29. Guardar resultados centrales


In [ ]:
STANDARD_OOF_PATH = (
    RESULTS /
    "kaki_virtual_sensor_standard_oof.csv"
)

SESSION_OOF_PATH_OUT = (
    RESULTS /
    "kaki_virtual_sensor_session_excluded_oof.csv"
)

SESSION_NULL_PATH = (
    RESULTS /
    "kaki_session_excluded_dummy_mean_oof.csv"
)

COMPARISON_PATH = (
    RESULTS /
    "kaki_virtual_sensor_comparison_metrics.csv"
)

standard_oof.to_csv(
    STANDARD_OOF_PATH,
    index=False
)

session_oof.to_csv(
    SESSION_OOF_PATH_OUT,
    index=False
)

session_null_oof.to_csv(
    SESSION_NULL_PATH,
    index=False
)

comparison_metrics.to_csv(
    COMPARISON_PATH,
    index=False
)

print("✅ Resultados guardados.")


# 30. Resumen científico automático


In [ ]:
def json_safe(
    obj
):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k):
                json_safe(
                    v
                )
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (
            list,
            tuple
        )
    ):
        return [
            json_safe(
                v
            )
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating
    ):
        return (
            None
            if np.isnan(
                obj
            )
            else float(
                obj
            )
        )

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray
    ):
        return json_safe(
            obj.tolist()
        )

    return obj


smallest_screen_payload = (
    None
    if smallest_engineering_panel is None
    else
    {
        k:
            json_safe(
                v
            )
        for k, v in smallest_engineering_panel.to_dict().items()
    }
)


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "camera":
        CAMERA,

    "virtual_filter_design": {
        "filter_shape":
            "Gaussian",

        "fwhm_values_nm":
            FWHM_VALUES,

        "candidate_center_step_nm":
            CENTER_STEP_NM,

        "k_values":
            K_VALUES,

        "forward_selection_metric":
            "inner fruit-balanced MAE",

        "selection_model":
            "RAW + StandardScaler + Ridge",

        "selection_alpha_grid":
            SELECTION_ALPHAS,

        "final_preprocessing":
            FINAL_PREPROCESSING,

        "final_alpha_grid":
            FINAL_ALPHAS,
    },

    "full249_standard":
        full_standard_metrics,

    "full249_session_excluded":
        full_session_metrics,

    "session_excluded_correct_null":
        session_null_metrics,

    "virtual_standard":
        standard_metrics.to_dict(
            orient="records"
        ),

    "virtual_session_excluded":
        session_metrics.to_dict(
            orient="records"
        ),

    "retention":
        retention.to_dict(
            orient="records"
        ),

    "paired_bootstrap":
        bootstrap_results.to_dict(
            orient="records"
        ),

    "best_virtual_standard":
        {
            k:
                json_safe(
                    v
                )
            for k, v in best_virtual_standard.to_dict().items()
        },

    "engineering_screen_delta_days":
        ENGINEERING_DELTA_DAYS,

    "smallest_panel_within_engineering_screen":
        smallest_screen_payload,

    "final_candidate_sensor_design": {
        "performance_claim":
            False,

        "fwhm_nm":
            best_fwhm,

        "k":
            best_k,

        "channels":
            final_candidate_panel.to_dict(
                orient="records"
            ),
    },

    "limitations": [
        (
            "All predictive performance remains internal to one experimental campaign."
        ),
        (
            "Outer test fruits are protected from band/filter selection, but the "
            "sensor design within each outer development set can observe all development days."
        ),
        (
            "The session-excluded test excludes the target day from model fitting "
            "but not from the earlier panel-design stage."
        ),
        (
            "The +0.10 day engineering screen is descriptive and is not a formal "
            "statistical equivalence test."
        ),
        (
            "The final candidate panel uses all 40 fruits after OOF evaluation and "
            "must be validated on an independent campaign before deployment claims."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_multivariate_virtual_sensor_design_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 31. Notas automáticas de revisión


In [ ]:
review_lines = [
    "KAKI — MULTIVARIATE VIRTUAL SENSOR DESIGN",
    "",
    f"Camera: {CAMERA}",
    f"Full bands: {len(wavelengths)}",
    f"Virtual FWHM values: {FWHM_VALUES}",
    f"k values: {K_VALUES}",
    "",
    (
        "Full249 standard FB-MAE = "
        f"{full_std_mae:.4f} d"
    ),
    (
        "Full249 session-excluded FB-MAE = "
        f"{full_sess_mae:.4f} d"
    ),
    (
        "Correct session-excluded null FB-MAE = "
        f"{null_sess_mae:.4f} d"
    ),
    "",
    "STANDARD VIRTUAL SENSORS:",
]

for _, r in standard_metrics.sort_values(
    [
        "fwhm_nm",
        "k"
    ]
).iterrows():

    review_lines.append(
        (
            f"  FWHM={int(r['fwhm_nm']):2d} nm, "
            f"k={int(r['k']):2d}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"delta-full={r['fruit_balanced_mae']-full_std_mae:+.4f}, "
            f"R2={r['r2']:.4f}"
        )
    )

review_lines.extend([
    "",
    "SESSION-EXCLUDED VIRTUAL SENSORS:",
])

for _, r in session_metrics.sort_values(
    [
        "fwhm_nm",
        "k"
    ]
).iterrows():

    review_lines.append(
        (
            f"  FWHM={int(r['fwhm_nm']):2d} nm, "
            f"k={int(r['k']):2d}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"delta-full={r['fruit_balanced_mae']-full_sess_mae:+.4f}, "
            f"improvement-null={null_sess_mae-r['fruit_balanced_mae']:+.4f}, "
            f"R2={r['r2']:.4f}"
        )
    )

review_lines.extend([
    "",
    (
        "Best virtual standard panel: "
        f"FWHM={int(best_virtual_standard['fwhm_nm'])} nm, "
        f"k={int(best_virtual_standard['k'])}, "
        f"FB-MAE={best_virtual_standard['fruit_balanced_mae']:.4f} d"
    ),
    (
        "Smallest panel within +0.10 d engineering screen: "
        f"{None if smallest_engineering_panel is None else int(smallest_engineering_panel['k'])}"
    ),
    "",
    (
        "Final all-data candidate sensor has NO independent performance estimate."
    ),
    (
        "Review this package before moving to CNN or deployment-oriented claims."
    ),
])

REVIEW_PATH = (
    RESULTS /
    "VIRTUAL_SENSOR_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
